# Helios — 01 Data Behavior Exploration

視覺探索 `daily_price` 資料的形狀、異常、行為模式。
與 `scripts/data_quality_report.py` 互補：報告告訴你「有什麼」，notebook 告訴你「長什麼樣」。

## 工作流程
1. 先跑 `uv run python scripts/download_daily.py --full` 抓 5 年資料
2. 跑 `uv run python scripts/data_quality_report.py` 看數字摘要
3. 打開這個 notebook 看視覺化、找 pattern
4. 把發現累積到 `docs/data_behavior_notes.md`

## 探索建議清單
- [ ] TAIEX 5 年走勢 + 200MA regime 標記
- [ ] 主要 ETF (0050/006208/0056/00878) 報酬率分布 vs TAIEX
- [ ] 除權息日的價格 gap 視覺化 (找 abnormal returns 的實際樣貌)
- [ ] 漲跌停日的群聚現象
- [ ] 月成交量趨勢 (流動性變化)
- [ ] 各 ETF 缺漏日是否同步 (應該完全同步)

In [ ]:
# 0. Setup
import os
import sys
from pathlib import Path

# 確保能 import helios 模組 (notebook 在 notebooks/ 目錄下)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)



# 可視化套件 (uv sync --group dev 才有)
import matplotlib.pyplot as plt
import polars as pl

from data.database import connect

plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# 1. 看 daily_price 整體狀況
with connect(read_only=True) as conn:
    df = conn.execute("""
        SELECT stock_id, COUNT(*) AS n_rows, MIN(date) AS first_d, MAX(date) AS last_d
        FROM daily_price
        GROUP BY stock_id
        ORDER BY n_rows DESC
    """).fetch_arrow_table()
pl.from_arrow(df)

In [ ]:
# 2. TAIEX 走勢
with connect(read_only=True) as conn:
    arrow = conn.execute(
        "SELECT date, close FROM daily_price WHERE stock_id = 'TAIEX' ORDER BY date"
    ).fetch_arrow_table()
taiex = pl.from_arrow(arrow)

if not taiex.is_empty():
    fig, ax = plt.subplots()
    ax.plot(taiex['date'], taiex['close'], linewidth=1, label='TAIEX close')
    # 加 200MA
    taiex_ma = taiex.with_columns(
        ma200=pl.col('close').rolling_mean(window_size=200)
    )
    ax.plot(taiex_ma['date'], taiex_ma['ma200'], linewidth=1, alpha=0.7, label='200MA')
    ax.set_title('TAIEX — 5 year price & 200MA regime baseline')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No TAIEX data yet. Run scripts/download_daily.py first.')

In [ ]:
# 3. 報酬率分布 — 看尾部多重
SYMBOL = '2330'
with connect(read_only=True) as conn:
    arrow = conn.execute(
        "SELECT date, close FROM daily_price WHERE stock_id = ? ORDER BY date",
        [SYMBOL],
    ).fetch_arrow_table()
df = pl.from_arrow(arrow)
if not df.is_empty():
    df = df.with_columns(pct=(pl.col('close') / pl.col('close').shift(1) - 1))
    pcts = df['pct'].drop_nulls().to_list()
    fig, ax = plt.subplots()
    ax.hist(pcts, bins=100)
    ax.axvline(0.105, color='red', linestyle='--', label='+10.5% (limit/dividend)')
    ax.axvline(-0.105, color='red', linestyle='--')
    ax.set_title(f'{SYMBOL} daily return distribution (n={len(pcts)})')
    ax.set_xlabel('Daily return')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 你的探索筆記

> 看到什麼有意思的 pattern，把它整理到 `docs/data_behavior_notes.md`

_(在這之下加 markdown / code cell 自由探索)_